# miniLISA TDI-2 Noise Modelling

This notebook builds second-generation TDI (X₂) combinations using the **miniLISA measurement model** from `miniLISA_TDI_modelling_copy.ipynb`.

Key differences from the standard LISA TDI-2 notebook:
- The η variables include **dual clock noise** sources: Moku phasemeter jitter `q_i(t)` and delay-board jitter `ε_i(t)`.
- The sideband measurements include a **shared reference-modulation** term (`ω_r^m`) coupling via both `q_ref` and `ε_i`.
- The delay operator uses **time-varying delays** `τ_{ij}(t) = τ_{ij}^0 + δτ_{ij}·t` as in the second notebook.
- Clock-noise correction via `r_{ij}` follows the same sideband-differencing approach.
- Board-jitter correction via REF variables follows the miniLISA prescription.

**Noise-term toggles** are collected in a single setup cell.

## Imports and display utilities

In [1]:
from sympy import Function, Symbol, symbols, simplify, latex, collect, expand
from IPython.display import display, Math
import re

def color_terms(latex_str):
    """Color-code noise terms in LaTeX output for readability."""
    # φ_REF → purple
    latex_str = re.sub(
        r'(\\phi_\{REF\}\{\\left\(.+?\\right\)\})',
        r'{\\color{purple} \1}', latex_str)
    # q_1 → red
    latex_str = re.sub(
        r'(q_\{([1])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{red} \1{\\left(\3\\right)}}', latex_str)
    # q_2 → orange
    latex_str = re.sub(
        r'(q_\{([2])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{orange} \1{\\left(\3\\right)}}', latex_str)
    # q_3 → yellow
    latex_str = re.sub(
        r'(q_\{([3])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{yellow} \1{\\left(\3\\right)}}', latex_str)
    # ε_A → cyan
    latex_str = re.sub(
        r'(\\epsilon_\{([A])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Cyan} \1{\\left(\3\\right)}}', latex_str)
    # ε_B → aquamarine
    latex_str = re.sub(
        r'(\\epsilon_\{([B])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Aquamarine} \1{\\left(\3\\right)}}', latex_str)
    # ε_C → spring green
    latex_str = re.sub(
        r'(\\epsilon_\{([C])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{SpringGreen} \1{\\left(\3\\right)}}', latex_str)
    return latex_str

def show(label, expr):
    display(Math(label + color_terms(latex(expr))))

## Symbols and noise functions

In [2]:
# ── Time variable ─────────────────────────────────────────────────────────────
t = Symbol('t')

# ── Delay parameters (static part) ────────────────────────────────────────────
tau12, tau21, tau13, tau31, tau23, tau32 = symbols(
    r'\tau_{12} \tau_{21} \tau_{13} \tau_{31} \tau_{23} \tau_{32}',
    real=True, positive=True)

# ── Delay rate-of-change (for time-varying delays τ_{ij}(t) = τ_{ij}^0 + δτ_{ij}·t) ─
dtau12, dtau21, dtau13, dtau31, dtau23, dtau32 = symbols(
    r'\dot\tau_{12} \dot\tau_{21} \dot\tau_{13} \dot\tau_{31} \dot\tau_{23} \dot\tau_{32}',
    real=True)

# ── Carrier frequencies ───────────────────────────────────────────────────────
omega1, omega2, omega3 = symbols(r'\omega_1 \omega_2 \omega_3', real=True)

# ── Modulation frequencies (per spacecraft) ───────────────────────────────────
omega1m, omega2m, omega3m = symbols(r'\omega_1^m \omega_2^m \omega_3^m', real=True)

# ── Shared reference-modulation frequency (miniLISA-specific) ─────────────────
omega_rm = Symbol(r'\omega_r^m', real=True)

# ── Laser phase noise ─────────────────────────────────────────────────────────
phi1   = Function(r'\phi_1')
phi2   = Function(r'\phi_2')
phi3   = Function(r'\phi_3')
phiREF = Function(r'\phi_{REF}')   # shared reference laser

# ── Moku phasemeter clock noise (LISA-like) ───────────────────────────────────
q1    = Function('q_1')
q2    = Function('q_2')
q3    = Function('q_3')
q_ref = Function('q_{ref}')   # reference modulation oscillator timing

# ── Delay-board clock noise (one board per spacecraft) ────────────────────────
epsilonA = Function(r'\epsilon_A')   # board A → delays received at SC1
epsilonB = Function(r'\epsilon_B')   # board B → delays received at SC2
epsilonC = Function(r'\epsilon_C')   # board C → delays received at SC3


# ── EOM modulation phase noise ─────────────────────────────────────────────────
N1_m = Function('P_{1}^{m}')
N2_m = Function('P_{2}^{m}')
N3_m = Function('P_{3}^{m}')

## Delay operator (time-varying)

The delay operator uses **time-varying delays** $\tau_{ij}(t) = \tau_{ij}^0 + \dot{\tau}_{ij}\cdot t$, consistent with the second notebook.

In [3]:
tau_dict = {
    (1,2): tau12, (2,1): tau21,
    (1,3): tau13, (3,1): tau31,
    (2,3): tau23, (3,2): tau32,
}

dtau_dict = {
    (1,2): dtau12, (2,1): dtau21,
    (1,3): dtau13, (3,1): dtau31,
    (2,3): dtau23, (3,2): dtau32,
}

def D(expr, tau_key):
    """
    Apply a single delay.
    - tau_key as tuple (i,j): uses time-varying τ_{ij}(t) = τ_{ij}^0 + δτ_{ij}·t
    - tau_key as a sympy symbol: applies it as a constant delay
    """
    if isinstance(tau_key, tuple):
        tau_val = tau_dict[tau_key] + dtau_dict[tau_key] * t
    else:
        tau_val = tau_key
    return expr.subs(t, t - tau_val)

def Dn(expr, *tau_keys):
    """
    Apply a sequence of delays left-to-right, each evaluated at the
    already-shifted time from all previous delays.

    Example:
        Dn(f, (1,2), (2,3))  →  f(t - τ₁₂(t) - τ₂₃(t - τ₁₂(t)))
    """
    for key in tau_keys:
        expr = D(expr, key)
    return expr

# ── Helper to drop δτ² terms (first-order in arm-rate changes) ────────────────
from sympy.core.function import AppliedUndef

def drop_dtau_squared(expr):
    """Linearise in δτ_{ij}: drop all terms of degree ≥ 2 in any δτ variable."""
    dtau_set = set(dtau_dict.values())
    def clean_arg(arg):
        result = 0
        for term in expand(arg).as_ordered_terms():
            deg = sum(term.as_powers_dict().get(dv, 0) for dv in dtau_set)
            if deg <= 1:
                result += term
        return result
    return expr.replace(
        lambda f: isinstance(f, AppliedUndef),
        lambda f: f.func(clean_arg(f.args[0]))
    )

## Noise toggles and measurement model

The per-link measurements $\eta_{ij}$, $\eta^{SB}_{ij}$, $\eta^{LSB}_{ij}$ are built from the **miniLISA model** (same physics as `miniLISA_TDI_modelling_copy.ipynb`).

### Carrier:
$$\eta_{ij} = \phi_j(t-\tau_{ij}) - \phi_i(t) - (\omega_j - \omega_i)\,q_i(t) + \omega_j\bigl[\epsilon_i(t) - \epsilon_i(t-\tau_{ij})\bigr]$$

### Upper sideband:
$$\eta^{SB}_{ij} = \phi_j(t-\tau_{ij}) - \phi_i(t) - (\omega_j + \omega_j^m - \omega_i - \omega_i^m)\,q_i(t) - \omega_i^m\,q_i(t) + (\omega_j^m + \omega_j)\bigl[\epsilon_i(t) - \epsilon_i(t-\tau_{ij})\bigr] + \omega_j^m\,D_{ij}[q_j] - \omega_r^m\bigl[D_{ij}[q_{\rm ref}] - q_{\rm ref}(t)\bigr] + \omega_r^m\bigl[D_{ij}[\epsilon_i] - \epsilon_i(t)\bigr]$$

### Clock-correcting variable:
$$r_{ij} = -\frac{\eta^{LSB}_{ij} - \eta^{SB}_{ij}}{2\,\omega_j^m}$$

In [4]:
# ── Noise-term toggles ─────────────────────────────────────────────────────────
include_phi           = False   # Laser phase noise φ_i
include_clock_noise   = True    # Moku phasemeter clock noise q_i
include_board_jitter  = True    # Delay-board clock noise ε_i=
include_modulation_noise = False  # EOM modulation phase noise P^m
include_ref_mod       = True    # Reference-modulation sideband coupling (ω_r^m)

# ── Spacecraft parameter map ───────────────────────────────────────────────────
sc = {
    1: (phi1, q1, omega1, omega1m, epsilonA, N1_m),
    2: (phi2, q2, omega2, omega2m, epsilonB, N2_m),
    3: (phi3, q3, omega3, omega3m, epsilonC, N3_m),
}


# ── Build all six one-way phase measurements ───────────────────────────────────
eta    = {}   # carrier
etaSB  = {}   # upper sideband
etaLSB = {}   # lower sideband
REF    = {}   # reference interferometer variable (for board-jitter correction)
r      = {}   # clock-correcting variable r_{ij}

for (i, j) in tau_dict:
    phi_i, q_i, om_i, omm_i, eps_i, N_i_m = sc[i]
    phi_j, q_j, om_j, omm_j, eps_j, N_j_m = sc[j]
    t_ij = (i, j)   # pass as tuple so D() uses time-varying delay

    # Laser phase terms
    phi_terms = (D(phi_j(t)) - (phi_i(t))) if include_phi else 0

    # Moku clock noise — carrier
    clock_C   = (-(om_j - om_i) * q_i(t)) if include_clock_noise else 0

    # Moku clock noise — upper sideband
    clock_SB  = (-(om_j - om_i + omm_j - omm_i) * q_i(t)
                 - omm_i * q_i(t)
                 + omm_j * D(q_j(t), t_ij)) if include_clock_noise else 0

    # Moku clock noise — lower sideband (sign flip on modulation part)
    clock_LSB = (-(-(om_i - om_j + omm_j - omm_i) * q_i(t)
                   - omm_i * q_i(t)
                   + omm_j * D(q_j(t), t_ij))) if include_clock_noise else 0

    # Delay-board jitter — carrier
    board_C   = (om_j * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0
    # Delay-board jitter — upper sideband
    board_SB  = ((om_j + omm_j) * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0
    # Delay-board jitter — lower sideband
    board_LSB = ((om_j - omm_j) * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0


    # EOM modulation phase noise
    mod_noise = (N_i_m(t) - D(N_j_m(t), t_ij)) if include_modulation_noise else 0

    # Reference-modulation coupling (miniLISA-specific)
    # ω_r^m·(q_ref(t−τ) − q_ref(t))  +  ω_r^m·(ε_i(t−τ) − ε_i(t))
    ref_mod_SB  = (
        -omega_rm * (D(q_ref(t), t_ij) - q_ref(t))
        + int(include_board_jitter) * omega_rm * (D(eps_i(t), t_ij) - eps_i(t))
    ) if include_ref_mod else 0
    ref_mod_LSB = (
        omega_rm * (D(q_ref(t), t_ij) - q_ref(t))
        - int(include_board_jitter) * omega_rm * (D(eps_i(t), t_ij) - eps_i(t))
    ) if include_ref_mod else 0

    # ── Assemble measurements ─────────────────────────────────────────────────
    eta[(i,j)] = phi_terms + clock_C + board_C

    etaSB[(i,j)]  = collect(expand(
        phi_terms + clock_SB + board_SB + mod_noise + ref_mod_SB
    ), [q_i(t), q_j(t), eps_i(t), D(q_j(t), t_ij), D(eps_i(t), t_ij)])

    etaLSB[(i,j)] = collect(expand(
        phi_terms + clock_LSB + board_LSB + mod_noise + ref_mod_LSB
    ), [q_i(t), q_j(t), eps_i(t), D(q_j(t), t_ij), D(eps_i(t), t_ij)])

    # REF variable — used for board-jitter correction
    REF[(i,j)] = (
        -int(include_clock_noise) * q_j(t)
        + int(include_board_jitter) * eps_i(t)
    )

    # Clock-correcting variable r_{ij} = −(η^{LSB} − η^{SB}) / (2 ω_j^m)
    r[(i,j)] = simplify(-(etaLSB[(i,j)] - etaSB[(i,j)]) / omm_j / 2)

## Inspect single-link measurements

In [5]:
for (i, j) in tau_dict:
    show(rf'\eta_{{{i}{j}}} = ', eta[(i,j)])
    show(rf'\eta^{{SB}}_{{{i}{j}}} = ', etaSB[(i,j)])
    show(rf'\eta^{{LSB}}_{{{i}{j}}} = ', etaLSB[(i,j)])
    show(rf'r_{{{i}{j}}} / \omega_{{{j}}}^m = ', r[(i,j)])
    print('─' * 60)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

────────────────────────────────────────────────────────────


...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

────────────────────────────────────────────────────────────


## Second-generation TDI — X₂ Michelson combination

The TDI-2 X combination on spacecraft 1 is built from path operators that account for the fact that arm lengths change linearly in time.

The path operators are defined so that the combination
$$X_2 = P_{12}(\eta_{12} + D_{12}\eta_{21}) + P_{13}(\eta_{13} + D_{13}\eta_{31})$$
cancels laser noise to second order in $\dot{\tau}$.

The operator definitions follow the second notebook:

$$P_{12}(x) = -\bigl(x - D_{131}x - D_{12131}x + D_{1312121}x\bigr)$$
$$P_{13}(x) = \phantom{-}\bigl(x - D_{121}x - D_{13121}x + D_{1213131}x\bigr)$$

In [6]:
# ── TDI-2 path operators on spacecraft 1 ──────────────────────────────────────
# Notation: D_131 means delay (3,1) then (1,3), etc.

def P12_X2(expr):
    return -(  expr
             - Dn(expr, (3,1),(1,3))
             - Dn(expr, (2,1),(1,2),(3,1),(1,3))
             + Dn(expr, (3,1),(1,3),(3,1),(1,3),(2,1),(1,2)))

def P13_X2(expr):
    return  (  expr
             - Dn(expr, (2,1),(1,2))
             - Dn(expr, (3,1),(1,3),(2,1),(1,2))
             + Dn(expr, (2,1),(1,2),(2,1),(1,2),(3,1),(1,3)))

# Delayed versions (already include the D_{12} / D_{13} first hop)
def P21_X2(expr): return P12_X2(Dn(expr, (1,2)))
def P31_X2(expr): return P13_X2(Dn(expr, (1,3)))

# Links not involving SC1 vanish in this X₂ channel
def P23_X2(expr): return 0
def P32_X2(expr): return 0

In [7]:
collect_vars = [q1(t), q2(t), q3(t), epsilonA(t), epsilonB(t), epsilonC(t),
                phi1(t), phi2(t), phi3(t)]

X2 = collect(expand(
      P12_X2(eta[(1,2)]) + P21_X2(eta[(2,1)])
    + P13_X2(eta[(1,3)]) + P31_X2(eta[(3,1)])
), collect_vars)

print("X₂ (uncorrected):")
show(r'X_2 = ', X2)

X₂ (uncorrected):


<IPython.core.display.Math object>

### Linearised in $\dot{\tau}$

Drop $\dot{\tau}^2$ terms (first-order in arm-rate changes).

In [8]:
#show(r'X_2^{\rm lin} = ', drop_dtau_squared(X2))

## Clock-noise correction for X₂

The correcting variables $R_{ij}$ are combinations of the round-trip sums
$$RT_{12} = r_{12} + D_{12}\,r_{21}, \qquad RT_{13} = r_{13} + D_{13}\,r_{31}$$

following equations (41a–f) of the standard TDI-2 derivation (see Tinto & Dhurandhar 2020 and the second notebook).

In [9]:
# Round-trip clock sums
RT12 = r[(1,2)] + Dn(r[(2,1)], (1,2))
RT13 = r[(1,3)] + Dn(r[(3,1)], (1,3))

R_X2 = {}

# (41a)
R_X2[(1,2)] = (
    - (RT12 - Dn(RT12, (3,1),(1,3)))
    + (2*RT13 - Dn(RT13, (2,1),(1,2)) - Dn(RT13, (3,1),(1,3),(2,1),(1,2)))
)

# (41b)
R_X2[(2,3)] = 0

# (41c)
R_X2[(3,1)] = (
    - (2*RT12 - Dn(RT12, (3,1),(1,3)) - Dn(RT12, (2,1),(1,2),(3,1),(1,3)))
    + (RT13 - Dn(RT13, (2,1),(1,2)))
    + (r[(1,3)]
       - Dn(r[(1,3)], (2,1),(1,2))
       - Dn(r[(1,3)], (3,1),(1,3),(2,1),(1,2))
       + Dn(r[(1,3)], (2,1),(1,2),(2,1),(1,2),(3,1),(1,3)))
)

# (41d)
R_X2[(2,1)] = (
    - (RT12 - Dn(RT12, (3,1),(1,3)))
    - (r[(1,2)]
       - Dn(r[(1,2)], (3,1),(1,3))
       - Dn(r[(1,2)], (2,1),(1,2),(3,1),(1,3))
       + Dn(r[(1,2)], (3,1),(1,3),(3,1),(1,3),(2,1),(1,2)))
    + (2*RT13 - Dn(RT13, (2,1),(1,2)) - Dn(RT13, (3,1),(1,3),(2,1),(1,2)))
)

# (41e)
R_X2[(3,2)] = 0

# (41f)
R_X2[(1,3)] = (
    - (2*RT12 - Dn(RT12, (3,1),(1,3)) - Dn(RT12, (2,1),(1,2),(3,1),(1,3)))
    + (RT13 - Dn(RT13, (2,1),(1,2)))
)

# Frequency difference coefficients
a = {(i,j): sc[i][2] - sc[j][2] for (i,j) in tau_dict}

# Sum correction over all triplets
correction_X2 = 0
for (i, j, k) in [(1,2,3), (2,3,1), (3,1,2)]:
    correction_X2 -= (-a[(i,j)] * R_X2[(i,j)] - a[(i,k)] * R_X2[(i,k)])

X2c = expand(X2 - correction_X2)

print("X₂ (Moku clock-noise corrected):")
show(r'X_2^{\rm corr} = ', X2c)

X₂ (Moku clock-noise corrected):


<IPython.core.display.Math object>

In [10]:
print("X₂ corrected, linearised in δτ:")
show(r'X_2^{\rm corr,\,lin} = ', drop_dtau_squared(X2c))

X₂ corrected, linearised in δτ:


<IPython.core.display.Math object>

## Delay-board jitter correction via REF interferometer

The REF variables encode cross-spacecraft reference-interferometer readouts and are used to subtract the remaining $\epsilon_i$ board jitter after Moku clock-noise correction.

$$\text{REF}_{AB} = \text{REF}_{13} - \text{REF}_{23}, \quad
  \text{REF}_{AC} = \text{REF}_{32} - \text{REF}_{12}, \quad
  \text{REF}_{CB} = \text{REF}_{21} - \text{REF}_{31}$$

The correction structure for TDI-2 mirrors the path-operator structure of $X_2$: each REF term is delayed along the same round-trip paths.

In [11]:
REF_AB = REF[(1,3)] - REF[(2,3)]
REF_AC = REF[(3,2)] - REF[(1,2)]
REF_CB = REF[(2,1)] - REF[(3,1)]

show(r'\mathrm{REF}_{AB} = ', REF_AB)
show(r'\mathrm{REF}_{AC} = ', REF_AC)
show(r'\mathrm{REF}_{CB} = ', REF_CB)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [12]:
# Board-jitter correction terms for X₂.
# These follow the same TDI-2 delay paths used in the path operators,
# extended by one arm compared to the TDI-1 correction.
Corr1 = omega1 * (
    - Dn(REF_AB, (1,2))
    + Dn(REF_AB, (2,1),(1,2))
    + Dn(REF_AB, (1,2),(3,1),(1,3))
    - Dn(REF_AB, (2,1),(1,2),(3,1),(1,3))
    + Dn(REF_AB, (1,2),(2,1),(1,2),(3,1),(1,3))
    - Dn(REF_AB, (2,1),(1,2),(2,1),(1,2),(3,1),(1,3))
    - Dn(REF_AB, (1,2),(3,1),(1,3),(3,1),(1,3),(2,1),(1,2))
)

Corr2 = omega1 * (
    - Dn(REF_AC, (1,3))
    + Dn(REF_AC, (3,1),(1,3))
    + Dn(REF_AC, (1,3),(2,1),(1,2))
    - Dn(REF_AC, (3,1),(1,3),(2,1),(1,2))
    + Dn(REF_AC, (1,3),(3,1),(1,3),(2,1),(1,2))
    - Dn(REF_AC, (3,1),(1,3),(3,1),(1,3),(2,1),(1,2))
    - Dn(REF_AC, (1,3),(2,1),(1,2),(2,1),(1,2),(3,1),(1,3))
)

Corr3 = omega1 * Dn(-REF_CB, (2,1),(1,2),(3,1),(1,3),(3,1),(1,3),(2,1),(1,2))

X2cc = expand(X2c + Corr1 + Corr2 + Corr3)

print("X₂ fully corrected (Moku clock + board jitter):")
show(r'X_2^{\rm full\,corr} = ', X2cc)

X₂ fully corrected (Moku clock + board jitter):


<IPython.core.display.Math object>

In [13]:
print("X₂ fully corrected, linearised in δτ:")
show(r'X_2^{\rm full\,corr,\,lin} = ', drop_dtau_squared(X2cc))

X₂ fully corrected, linearised in δτ:


<IPython.core.display.Math object>

## Summary

| Variable | Description |
|---|---|
| `X2` | Raw TDI-2 Michelson on SC1, uncorrected |
| `X2c` | After Moku clock-noise (`q_i`) correction via `r_{ij}` |
| `X2cc` | After additional delay-board jitter (`ε_i`) correction via REF variables |

The dominant residuals in `X2cc` (with current toggle settings) come from the **reference-modulation** (`ω_r^m`) cross-terms that are not fully suppressed by the sideband-differencing `r_{ij}` variable — these are a miniLISA-specific effect absent in the standard LISA TDI-2 model.